In [1]:
from typing import Annotated
import operator
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI # Ensure this is installed
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import (SystemMessagePromptTemplate, 
                                    HumanMessagePromptTemplate, 
                                    AIMessagePromptTemplate, 
                                    ChatPromptTemplate,
                                    FewShotChatMessagePromptTemplate)



In [2]:
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", 
                               max_tokens=100,
                               temperature=0,
                               model_kwargs={'seed' :365}
                               )


/opt/venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [3]:
TEMPLATE_AI = '''{response}'''
TEMPLATE_H = '''
Human: i recently adopted a {pet}.
Could you suggest some {pet} names?
'''

In [4]:
from langchain_core.prompts import PromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, AIMessagePromptTemplate, ChatPromptTemplate

message_template_ai  = AIMessagePromptTemplate.from_template(template=TEMPLATE_AI)
message_template_h  = HumanMessagePromptTemplate.from_template(template=TEMPLATE_H)
example_template = ChatPromptTemplate.from_messages([message_template_ai, message_template_h])



In [5]:
examples = [{'pet': 'dog', 'response': 'Here are some dog name suggestions: Max, Bella, Charlie, Luna, Rocky, Daisy, Buddy, Sadie, Cooper, Molly.'},
            {'pet': 'cat', 'response': 'Here are some cat name suggestions: Whiskers, Luna, Oliver, Bella, Simba, Chloe, Leo, Daisy, Max, Nala.'},
]

In [6]:
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_template)

In [7]:
chat_template = ChatPromptTemplate.from_messages([few_shot_prompt, message_template_h])

In [9]:
chat_value = chat_template.invoke({'pet': 'rabbit'})

In [11]:
for i in chat_value.messages:
    print(i.content)

Here are some dog name suggestions: Max, Bella, Charlie, Luna, Rocky, Daisy, Buddy, Sadie, Cooper, Molly.

Human: i recently adopted a dog.
Could you suggest some dog names?

Here are some cat name suggestions: Whiskers, Luna, Oliver, Bella, Simba, Chloe, Leo, Daisy, Max, Nala.

Human: i recently adopted a cat.
Could you suggest some cat names?


Human: i recently adopted a rabbit.
Could you suggest some rabbit names?



In [10]:
model.invoke(chat_value)

AIMessage(content=[{'type': 'text', 'text': 'Congratulations on your new rabbit! Here are some name suggestions for a bunny, ranging from cute to classic:\n\n**Cute & Sweet:**\n*   Thumper\n*   Bun-Bun\n*   Clover\n*   Marshmallow\n*   Pip\n*   Button\n*   Honey\n\n**Food-Inspired:**\n*   Basil\n*   Mochi\n*   Peanut\n*   Biscuit\n*   Ginger\n*   Olive\n*   Pumpkin', 'extras': {'signature': 'EnMKcQFpFH0T/Vyc65hGEDGqaBozwmpl/IBXpViiBhgKhKqjEsWAIRWfEDHAabIWpX/rqBul5Fs5ITNr2JiQMFwTxJycOdlrmB4U3BGW75FFQTB5rloavtpl0TxZ59yFoogcbuOQV3zNO9Xp6LdxZUC+wfeZ'}}], additional_kwargs={}, response_metadata={'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a166f9-096a-7351-81a2-5bb819427bab-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 116, 'output_tokens': 96, 'total_tokens': 212, 'input_token_details': {'cache_read': 0}})